# N03 — Embeddings: quando o texto vira geometria

**Capítulo-âncora:** C5 — Embeddings · **Invariante:** Inv. 3 (Camada Dupla)

## O que este notebook resolve

Embedding é o conceito que sustenta RAG (C6), busca semântica, clusterização de feedback, deduplicação inteligente, e dezenas de outras aplicações práticas. Em texto, a definição soa abstrata: 'representação vetorial densa do significado'. **Na tela, vira concreto em uma linha:** frases parecidas ficam próximas no espaço; frases diferentes ficam longe.

Este notebook gera embeddings reais de oito frases brasileiras, calcula similaridades, plota o resultado em 2D, e roda uma busca semântica simples. Em quinze minutos, você vê na própria máquina o que a teoria descreve.

**Nota:** Anthropic não oferece API pública de embeddings — o padrão de mercado para este notebook é usar Voyage AI, OpenAI text-embedding-3-small, ou modelo open-source via sentence-transformers. Para manter execução simples e offline, usamos `sentence-transformers` com modelo multilíngue local. Substitua pela API que você adotar.


## Setup


In [ ]:
# sentence-transformers roda localmente e suporta português.
# Instalação no requirements.txt. Modelo cabe em ~120MB.
try:
    from sentence_transformers import SentenceTransformer
    LOCAL_OK = True
except ImportError:
    print('sentence-transformers não instalado.')
    print('Instale com: pip install sentence-transformers')
    LOCAL_OK = False

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

if LOCAL_OK:
    # Multilíngue, leve, bom em PT.
    modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print(f'Modelo carregado. Dimensão dos embeddings: {modelo.get_sentence_embedding_dimension()}')


## Experimento 1 — Frases parecidas geram vetores parecidos


In [ ]:
frases = [
    'O cliente cancelou a assinatura ontem.',
    'A subscription do usuário foi encerrada.',
    'Houve um cancelamento da nossa solução SaaS.',
    'O time precisa aprovar o orçamento da próxima sprint.',
    'A diretoria vai discutir o budget do próximo ciclo.',
    'Decidimos investir em uma nova frente de IA generativa.',
    'Vamos alocar recursos para um projeto de inteligência artificial.',
    'O café da máquina do escritório está acabando.',
]

if LOCAL_OK:
    vetores = modelo.encode(frases)
    print(f'Shape: {vetores.shape}  → {len(frases)} frases × {vetores.shape[1]} dimensões')
    print(f'Cada vetor tem {vetores.shape[1]} números entre -1 e 1.')


## Experimento 2 — Similaridade coseno (a regra geométrica de busca semântica)


In [ ]:
def similaridade_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

if LOCAL_OK:
    n = len(frases)
    matriz = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            matriz[i, j] = similaridade_coseno(vetores[i], vetores[j])
    
    print('Matriz de similaridade (1.00 = igual; 0.00 = ortogonal; -1.00 = oposto)')
    print('     ' + '  '.join(f'F{i}' for i in range(n)))
    for i in range(n):
        linha = '  '.join(f'{matriz[i,j]:.2f}' for j in range(n))
        print(f'F{i}:  {linha}')


**Leia a matriz:** as frases 0, 1, 2 (cancelamento em variações) tendem a ter similaridade alta entre si (>0.6). As frases 5 e 6 (IA generativa, vocabulário próximo) também. A frase 7 (café) deve ter similaridade baixa com todas — está semanticamente isolada. **Esse é o motor por trás de busca semântica, RAG, deduplicação inteligente.**


## Experimento 3 — Visualizar o espaço em 2D


In [ ]:
if LOCAL_OK:
    pca = PCA(n_components=2)
    vetores_2d = pca.fit_transform(vetores)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    cores = ['red'] * 3 + ['blue'] * 2 + ['green'] * 2 + ['purple']
    for i, (frase, ponto, cor) in enumerate(zip(frases, vetores_2d, cores)):
        ax.scatter(ponto[0], ponto[1], c=cor, s=200)
        ax.annotate(f'F{i}', (ponto[0], ponto[1]), xytext=(5, 5),
                    textcoords='offset points', fontsize=11, fontweight='bold')
    ax.set_xlabel('Componente principal 1')
    ax.set_ylabel('Componente principal 2')
    ax.set_title('Frases projetadas em 2D — clusters de significado')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print()
    print('Legenda:')
    for i, frase in enumerate(frases):
        print(f'  F{i}: {frase}')


Os pontos vermelhos (cancelamento) devem ficar juntos. Os azuis (orçamento) também. Os verdes (IA) também. O roxo (café) fica isolado. **PCA reduz 384 dimensões para 2 para o olho enxergar — sempre há perda, mas o agrupamento sobrevive.** É a evidência geométrica que sustenta RAG.


## Experimento 4 — Busca semântica mínima


In [ ]:
if LOCAL_OK:
    def buscar(consulta: str, top_k: int = 3):
        vetor_consulta = modelo.encode([consulta])[0]
        sims = [similaridade_coseno(vetor_consulta, v) for v in vetores]
        ranking = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)[:top_k]
        print(f'Consulta: {consulta!r}')
        for rank, (idx, sim) in enumerate(ranking, 1):
            print(f'  {rank}. (sim={sim:.3f}) {frases[idx]}')
        print()
    
    buscar('Algum cliente deixou a plataforma?')
    buscar('Decisão sobre investimento')
    buscar('algo sobre a copa de café')


**Você acabou de implementar busca semântica em três linhas.** Em produção, troca-se o `for` por um índice vetorial (FAISS, pgvector, Pinecone, Chroma) e o modelo local por uma API de embeddings de qualidade superior. Mas o algoritmo é exatamente esse.


## O que fazer com isso

1. **Antes de adotar RAG, prototipe a busca semântica com sentence-transformers local em duas horas.** Se a relevância dos top-k for ruim no protótipo, RAG completo não vai consertar.
2. **Toda vez que você for clusterizar feedback, deduplicar texto, ou montar busca de FAQ, embeddings é a primeira opção.** Custa centavos, roda em segundos.
3. **Lembre-se da Camada Dupla:** o modelo de embedding pode mudar. O conceito de espaço vetorial e similaridade coseno é o que dura.

**Próximo notebook:** [N04 — Prompt Caching](./n04-prompt-caching.ipynb), onde o custo composto vira economia mensurável.
